# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.
- Load nyc taxi data from January of 2025 and compare data.
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

---

## How this copy is organised

**Lab 03 — ECE Big Data Processing (Adaltas), lesson 03 *SparkSQL & DataFrames*.**

- **Main language: the PySpark DataFrame API** (Parts 1 and 2). **Part 3 re-answers three questions in pure Spark SQL**, two of them with a join, and cross-checks the two answers programmatically.
- Every question gets a code cell and, below it, a short interpretation of what the output shows.
- Data files are downloaded **once** into `data/` next to this notebook and reused on later runs. The extra 2019 months used in the last section go to a separate directory, to keep the lab folder light.
- Environment: `quay.io/jupyter/pyspark-notebook` (Spark 4.2, Python 3.13) + `plotly` for Spark 4's native DataFrame plotting.

## 0. Setup

In [1]:
import time
from pathlib import Path

import requests
import plotly.io as pio
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window

# Plotly figures: JSON mimetype for JupyterLab + a CDN-backed HTML fallback for other viewers
pio.renderers.default = "plotly_mimetype+notebook_connected"

# start a spark session and create a spark context
spark = (SparkSession.builder
         .appName("nyc_taxi")
         .config("spark.driver.memory", "6g")                       # a cached 7.7 M-row DataFrame + a 84 M-row scan in 4.1
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # fast toPandas() for the plots
         .config("spark.ui.showConsoleProgress", "false")            # no progress bars in the saved outputs
         .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")  # keep the (unused) warehouse folder out of the lab dir
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"Spark {spark.version} | {sc.defaultParallelism} cores | driver memory {spark.conf.get('spark.driver.memory')}")

Spark 4.2.0 | 20 cores | driver memory 6g


### Data download

The handed-out cell downloads the file with `requests` on every run. The helper below keeps the same idea but is **idempotent**: a file already present in `data/` is not downloaded again, and a bad HTTP status raises instead of silently writing an error page to disk.

In [2]:
DATA_DIR = Path("data")
BASE_URL = "https://d37ci6vzurychx.cloudfront.net"


def download(remote_path: str, local_dir: Path = DATA_DIR) -> Path:
    """Fetch a TLC file once; later runs reuse the local copy."""
    target = local_dir / Path(remote_path).name
    if target.exists() and target.stat().st_size > 0:
        return target
    local_dir.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    response = requests.get(f"{BASE_URL}/{remote_path}", timeout=600)
    response.raise_for_status()
    target.write_bytes(response.content)
    print(f"downloaded {target.name}: {target.stat().st_size / 1e6:.1f} MB in {time.time() - t0:.0f} s")
    return target


# set dl url for January 2019 trip data and get the data
jan_2019_trip_data = download("trip-data/yellow_tripdata_2019-01.parquet")
print(jan_2019_trip_data, f"{jan_2019_trip_data.stat().st_size / 1e6:.1f} MB")

data/yellow_tripdata_2019-01.parquet 110.4 MB


In [3]:
# create the dataframe
df_trips = spark.read.parquet(str(jan_2019_trip_data))

df_trips.printSchema()
print(f"{df_trips.count():,} rows in {df_trips.rdd.getNumPartitions()} partitions")

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: integer (nullable = true)



7,696,617 rows in 20 partitions


# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

> Note on the schema printed above: the two timestamps are `timestamp_ntz` (no time zone), so Spark will **not** shift them to the session time zone — hours of the day are New York local time as recorded by the meter. `passenger_count` and `RatecodeID` are stored as `double` in the 2019 file (they become integers in later years, see Part 2).

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

**Language used for Part 1 and Part 2: the PySpark DataFrame API.** Spark SQL is used in Part 3.

### 1.0 Before answering: what is actually in the "January 2019" file?

Two quick checks that change how the later questions must be answered: the range of pickup timestamps, and whether the file contains exact duplicate rows (which matters for the choice of a unique key).

In [5]:
in_jan_2019 = (F.col("tpep_pickup_datetime") >= "2019-01-01") & (F.col("tpep_pickup_datetime") < "2019-02-01")

df_trips.select(
    F.min("tpep_pickup_datetime").alias("first_pickup"),
    F.max("tpep_pickup_datetime").alias("last_pickup"),
    F.sum((~in_jan_2019).cast("int")).alias("pickups_outside_jan_2019"),
).show(truncate=False)

n_rows = df_trips.count()
n_distinct = df_trips.distinct().count()
print(f"exact duplicate rows: {n_rows - n_distinct:,} of {n_rows:,}")

+-------------------+-------------------+------------------------+
|first_pickup       |last_pickup        |pickups_outside_jan_2019|
+-------------------+-------------------+------------------------+
|2001-02-02 14:55:07|2088-01-24 00:25:39|537                     |
+-------------------+-------------------+------------------------+



exact duplicate rows: 0 of 7,696,617


**What this changes.** The "January 2019" file holds **537 pickups outside January 2019**, dated from 2001 to… 2088 — meter clocks set wrong, or records uploaded late. That is 0.007 % of the rows, harmless for averages, but fatal for questions such as *"which single day was the slowest?"* (a day in 2008 with three trips would win). Time-based questions are therefore answered on the 7,696,080 pickups that really fall in January 2019, while record-level questions (highest passenger count, highest `extra`…) keep every row. There are **no exact duplicate rows**.

### 1.1 A unique key for each record

The raw file has no primary key. This file happens to contain no exact duplicate row, but nothing in the data model prevents two distinct trips from having identical values in every column, so hashing the columns (`sha2(concat_ws(...))`) is not a safe key in general. `monotonically_increasing_id()` instead assigns a 64-bit id from the partition index (upper 31 bits) and the row position inside the partition: **unique, but neither consecutive nor stable across recomputations**. We therefore `cache()` the DataFrame and materialise it right away, so that `trip_id` stays fixed for the rest of the session.

The same cell adds the derived columns needed by most later questions (pickup date, hour, day of week, trip duration). `timestamp_diff` (Spark 4) works directly on `timestamp_ntz`.

In [6]:
df = (df_trips
      .withColumn("trip_id", F.monotonically_increasing_id())
      .withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
      .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
      .withColumn("pickup_dow", F.date_format("tpep_pickup_datetime", "EEEE"))
      # dayofweek() is 1 = Sunday ... 7 = Saturday; remap to 1 = Monday ... 7 = Sunday for sorting
      .withColumn("pickup_dow_num", (F.dayofweek("tpep_pickup_datetime") + 5) % 7 + 1)
      .withColumn("duration_min",
                  F.timestamp_diff("SECOND", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")) / 60.0)
      # average speed; the when() guard avoids a division by zero (Spark 4 runs in ANSI mode and would raise)
      .withColumn("avg_mph", F.when(F.col("duration_min") > 0, F.col("trip_distance") / (F.col("duration_min") / 60.0)))
      .cache())

df.count()  # materialise the cache: trip_id is now fixed for the session

df.select("trip_id", "tpep_pickup_datetime", "pickup_date", "pickup_hour", "pickup_dow", "pickup_dow_num", "duration_min", "avg_mph").show(5)
df.agg(F.count("*").alias("rows"), F.countDistinct("trip_id").alias("distinct_trip_ids")).show()

+-----------+--------------------+-----------+-----------+----------+--------------+------------------+------------------+
|    trip_id|tpep_pickup_datetime|pickup_date|pickup_hour|pickup_dow|pickup_dow_num|      duration_min|           avg_mph|
+-----------+--------------------+-----------+-----------+----------+--------------+------------------+------------------+
|77309411328| 2019-01-01 00:46:40| 2019-01-01|          0|   Tuesday|             2| 6.666666666666667|13.499999999999998|
|77309411329| 2019-01-01 00:59:47| 2019-01-01|          0|   Tuesday|             2|              19.2|             8.125|
|77309411330| 2018-12-21 13:48:30| 2018-12-21|         13|    Friday|             5| 4.166666666666667|               0.0|
|77309411331| 2018-11-28 15:52:25| 2018-11-28|         15| Wednesday|             3|3.3333333333333335|               0.0|
|77309411332| 2018-11-28 15:56:57| 2018-11-28|         15| Wednesday|             3|               1.6|               0.0|
+-----------+---

+-------+-----------------+
|   rows|distinct_trip_ids|
+-------+-----------------+
|7696617|          7696617|
+-------+-----------------+



In [7]:
# Time-based questions ("busiest day", "busiest hour"...) are answered on January 2019 only,
# otherwise a stray 2008 timestamp with three trips would be "the slowest day".
jan = df.filter(in_jan_2019)
print(f"January 2019 pickups kept for the time-based questions: {jan.count():,}")

January 2019 pickups kept for the time-based questions: 7,696,080


### 1.2 Which trip has the highest passenger count?

In [8]:
df.groupBy("passenger_count").count().orderBy("passenger_count").show()

trip_cols = ["trip_id", "tpep_pickup_datetime", "passenger_count", "trip_distance",
             "fare_amount", "total_amount", "PULocationID", "DOLocationID"]
(df.orderBy(F.desc("passenger_count"), F.desc("total_amount"))
   .select(*trip_cols)
   .show(5, truncate=False))

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL|  28672|
|            0.0| 117381|
|            1.0|5456515|
|            2.0|1113894|
|            3.0| 314692|
|            4.0| 140753|
|            5.0| 323842|
|            6.0| 200811|
|            7.0|     19|
|            8.0|     29|
|            9.0|      9|
+---------------+-------+



+-----------+--------------------+---------------+-------------+-----------+------------+------------+------------+
|trip_id    |tpep_pickup_datetime|passenger_count|trip_distance|fare_amount|total_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------+-------------+-----------+------------+------------+------------+
|77313946035|2019-01-19 16:45:25 |9.0            |0.0          |92.0       |110.76      |1           |1           |
|77316785204|2019-01-30 22:17:51 |9.0            |13.38        |90.0       |90.8        |162         |265         |
|77314263553|2019-01-21 03:46:51 |9.0            |0.0          |9.0        |12.74       |264         |230         |
|77310361284|2019-01-05 13:12:29 |9.0            |0.0          |9.8        |12.6        |68          |68          |
|77312295323|2019-01-13 04:13:24 |9.0            |0.0          |9.0        |12.25       |263         |263         |
+-----------+--------------------+---------------+-------------+--------

**Answer.** The highest passenger count is **9**, and it is a **tie between 9 trips** (plus 29 trips with 8 and 19 with 7 passengers). Sorted by amount, the first is trip `77313946035` (19 January, 16:45, Newark Airport → Newark Airport, 0 miles, $92 fare) — clearly not a real 9-person ride. The second, `77316785204` (30 January, 22:17, Midtown East → outside NYC, 13.4 miles, $90), looks like a genuine long ride.

A yellow cab is licensed for 4 passengers (5 in a minivan, plus one child), so 7–9 is physically impossible: `passenger_count` is typed in by the driver, and these 57 records are keying errors. Two other things the distribution shows: **117,381 trips with 0 passengers** (1.5 %, the driver skipped the field) and **28,672 nulls**, and 71 % of trips carry a single passenger.

### 1.3 What is the average passenger count?

In [9]:
df.agg(
    F.round(F.avg("passenger_count"), 3).alias("avg_all_rows"),
    F.round(F.avg(F.when(F.col("passenger_count") > 0, F.col("passenger_count"))), 3).alias("avg_excluding_zero"),
    F.sum((F.col("passenger_count") == 0).cast("int")).alias("rows_with_zero_passengers"),
    F.sum(F.col("passenger_count").isNull().cast("int")).alias("rows_with_null"),
).show()

+------------+------------------+-------------------------+--------------+
|avg_all_rows|avg_excluding_zero|rows_with_zero_passengers|rows_with_null|
+------------+------------------+-------------------------+--------------+
|       1.567|             1.591|                   117381|         28672|
+------------+------------------+-------------------------+--------------+



**Answer: 1.57 passengers per trip** (1.59 if the 117,381 "0 passenger" rows, which cannot be real, are excluded; the 28,672 nulls are ignored by `avg` automatically). The number is dominated by solo riders (71 % of trips), with a secondary bump at 5–6 passengers (minivans).

### 1.4 Shortest / longest trip by distance and by time

Raw extremes first, then the same question restricted to *plausible* trips (positive distance, between 1 minute and 24 hours, average speed at most 80 mph), because the raw extremes are all data errors.

In [10]:
ext_cols = ["trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
            "duration_min", "avg_mph", "fare_amount", "PULocationID", "DOLocationID"]

print("Shortest by distance");  df.orderBy("trip_distance").select(*ext_cols).show(3)
print("Longest by distance");   df.orderBy(F.desc("trip_distance")).select(*ext_cols).show(3)
print("Shortest by time");      df.orderBy("duration_min").select(*ext_cols).show(3)
print("Longest by time");       df.orderBy(F.desc("duration_min")).select(*ext_cols).show(3)

print(f"trips with distance = 0: {df.filter('trip_distance = 0').count():,}   "
      f"trips with duration <= 0: {df.filter('duration_min <= 0').count():,}")

Shortest by distance


+-----------+--------------------+---------------------+-------------+------------------+-------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+-------+-----------+------------+------------+
|77309411332| 2018-11-28 15:56:57|  2018-11-28 15:58:33|          0.0|               1.6|    0.0|       52.0|         193|         193|
|77309411333| 2018-11-28 16:25:49|  2018-11-28 16:28:26|          0.0|2.6166666666666667|    0.0|        3.5|         193|         193|
|77309411334| 2018-11-28 16:29:37|  2018-11-28 16:33:43|          0.0|               4.1|    0.0|       52.0|         193|         193|
+-----------+--------------------+---------------------+-------------+------------------+-------+-----------+------------+------------+
only showing top 3 rows
Longest by distance


+-----------+--------------------+---------------------+-------------+-----------------+------------------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|     duration_min|           avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+-----------------+------------------+-----------+------------+------------+
|77315485419| 2019-01-25 21:56:39|  2019-01-25 22:06:08|        831.8|9.483333333333333| 5262.706502636204|        8.5|         140|         239|
|77313697961| 2019-01-18 16:32:24|  2019-01-18 16:39:20|        700.7|6.933333333333334|           6063.75|        6.0|         236|         262|
|77316182313| 2019-01-28 17:24:11|  2019-01-29 04:37:16|       214.01|673.0833333333334|19.077281168750773|      760.0|         265|         265|
+-----------+--------------------+---------------------+-------------+-----------------+------------------+-----------+-----

+-----------+--------------------+---------------------+-------------+-------------------+-------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|       duration_min|avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+-------------------+-------+-----------+------------+------------+
|77310614512| 2019-01-06 15:15:08|  2018-11-09 02:34:38|          3.3|           -84280.5|   NULL|       16.0|         100|          79|
|77312387621| 2019-01-13 15:15:27|  2018-12-25 07:37:43|          7.6|-27817.733333333334|   NULL|       29.5|         164|         145|
|77314153074| 2019-01-20 15:15:30|  2019-01-18 15:57:16|          1.9| -2838.233333333333|   NULL|       15.5|         186|         162|
+-----------+--------------------+---------------------+-------------+-------------------+-------+-----------+------------+------------+
only showing top 3 rows
Longest by time


+-----------+--------------------+---------------------+-------------+------------------+--------------------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|             avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+--------------------+-----------+------------+------------+
|77309479595| 2019-01-01 07:01:20|  2019-01-31 14:29:21|          1.2| 43648.01666666667|0.001649559487429...|        6.5|          48|         163|
|77310003590| 2019-01-03 22:24:36|  2019-01-27 10:41:17|          1.1|33856.683333333334|0.001949393546621...|        8.5|          50|         170|
|77310287184| 2019-01-05 04:21:40|  2019-01-27 01:53:46|          3.9|           31532.1|0.007421009066950822|       16.5|         114|         237|
+-----------+--------------------+---------------------+-------------+------------------+-----------------

trips with distance = 0: 55,089   trips with duration <= 0: 6,557


In [11]:
plausible = df.filter((F.col("trip_distance") > 0) & (F.col("duration_min").between(1, 24 * 60)) & (F.col("avg_mph") <= 80))
print(f"plausible trips: {plausible.count():,} of {df.count():,}")
print(f"trips lasting 23 h or more (drop-off stamped one day late?): {df.filter('duration_min >= 23 * 60').count():,}")

print("Longest plausible trip by distance"); plausible.orderBy(F.desc("trip_distance")).select(*ext_cols).show(3)
print("Longest plausible trip by time");     plausible.orderBy(F.desc("duration_min")).select(*ext_cols).show(3)
print("Shortest plausible trip by distance"); plausible.orderBy("trip_distance", "duration_min").select(*ext_cols).show(3)

plausible trips: 7,610,074 of 7,696,617
trips lasting 23 h or more (drop-off stamped one day late?): 18,744
Longest plausible trip by distance


+-----------+--------------------+---------------------+-------------+------------------+------------------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|           avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+------------------+-----------+------------+------------+
|77316182313| 2019-01-28 17:24:11|  2019-01-29 04:37:16|       214.01| 673.0833333333334|19.077281168750773|      760.0|         265|         265|
|77314118862| 2019-01-20 12:22:24|  2019-01-20 17:05:36|       211.36|             283.2|44.779661016949156|       50.0|         132|         265|
|77314293113| 2019-01-21 10:13:05|  2019-01-21 13:28:01|       201.27|194.93333333333334|61.950410396716826|      150.0|         208|         265|
+-----------+--------------------+---------------------+-------------+------------------+------------------+----------

+-----------+--------------------+---------------------+-------------+------------------+------------------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|      duration_min|           avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------------+------------------+-----------+------------+------------+
|77312722469| 2019-01-14 23:24:05|  2019-01-15 23:24:04|        18.61|1439.9833333333333|0.7754256415004803|       52.0|         132|         125|
|77316946102| 2019-01-31 14:20:52|  2019-02-01 14:20:50|         7.39|1439.9666666666667|0.3079237945322808|       39.5|         163|         138|
|77315998473| 2019-01-27 20:52:56|  2019-01-28 20:52:51|        19.68|1439.9166666666667| 0.820047456450026|       52.0|         132|         107|
+-----------+--------------------+---------------------+-------------+------------------+------------------+----------

+-----------+--------------------+---------------------+-------------+------------+-------+-----------+------------+------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|duration_min|avg_mph|fare_amount|PULocationID|DOLocationID|
+-----------+--------------------+---------------------+-------------+------------+-------+-----------+------------+------------+
|77309669295| 2019-01-02 12:50:55|  2019-01-02 12:51:55|         0.01|         1.0|    0.6|        2.5|         114|         114|
|77309749612| 2019-01-02 18:12:52|  2019-01-02 18:13:52|         0.01|         1.0|    0.6|       52.0|         132|         132|
|77310491889| 2019-01-05 23:19:01|  2019-01-05 23:20:01|         0.01|         1.0|    0.6|        2.5|         107|         107|
+-----------+--------------------+---------------------+-------------+------------+-------+-----------+------------+------------+
only showing top 3 rows


**Raw extremes are all data errors**, which is the real lesson of this question:

- **Shortest by distance: 0.0 miles**, shared by **55,089 trips** (0.7 %) — meter started and reset, or GPS failure. The shortest *plausible* trip is 0.01 mile in exactly one minute, i.e. a passenger who changed their mind.
- **Longest by distance: 831.8 miles… in 9.5 minutes, for $8.50** (trip `77315485419`), followed by 700.7 miles in 7 minutes. Only the odometer field is wrong: time and fare describe a normal 2-mile ride. Once a speed check (≤ 80 mph) is added, the longest trip is `77316182313`: **214 miles in 11.2 hours for $760**, pickup and drop-off both "unknown" — an out-of-town negotiated ride.
- **Shortest by time: −84,280 minutes** — a drop-off stamped two months *before* the pickup; 4 trips have a negative duration and **6,557 have a duration ≤ 0**.
- **Longest by time: 43,648 minutes = 30.3 days** for a 1.2-mile, $6.50 ride: the drop-off timestamp is wrong, not the trip. Among plausible trips the longest ones all last **1,439.9 minutes, i.e. 24 hours minus a few seconds**, and there are 18,744 trips of 23 h or more: a systematic artefact where the drop-off is stamped one day late. The genuinely longest ride in time is again the 214-mile one.

The `plausible` filter (positive distance, 1 min – 24 h, ≤ 80 mph) keeps 7,610,074 trips, i.e. 98.9 % of the file: the errors are rare, but they own every extreme.

### 1.5 Busiest and slowest single day

In [12]:
per_day = jan.groupBy("pickup_date", "pickup_dow").agg(F.count("*").alias("trips"))

print("Busiest days"); per_day.orderBy(F.desc("trips")).show(5)
print("Slowest days"); per_day.orderBy("trips").show(5)

Busiest days


+-----------+----------+------+
|pickup_date|pickup_dow| trips|
+-----------+----------+------+
| 2019-01-25|    Friday|292499|
| 2019-01-11|    Friday|291714|
| 2019-01-31|  Thursday|284625|
| 2019-01-17|  Thursday|284580|
| 2019-01-24|  Thursday|281959|
+-----------+----------+------+
only showing top 5 rows
Slowest days


+-----------+----------+------+
|pickup_date|pickup_dow| trips|
+-----------+----------+------+
| 2019-01-01|   Tuesday|189432|
| 2019-01-21|    Monday|192826|
| 2019-01-02| Wednesday|198737|
| 2019-01-20|    Sunday|203114|
| 2019-01-06|    Sunday|208823|
+-----------+----------+------+
only showing top 5 rows


**Busiest day: Friday 25 January (292,499 trips)**, a whisker ahead of Friday 11 January (291,714); the next three are Thursdays. **Slowest day: Tuesday 1 January (189,432)**, New Year's Day, followed by **Monday 21 January (192,826)** — Martin Luther King Day, which was also the coldest morning of the winter after the 19–20 January snow-and-ice storm — then 2 January and the two Sundays of the 6th and 20th. Holidays and weather explain the troughs; end-of-week evenings explain the peaks.

### 1.6 Busiest and slowest time of day

Two granularities: by hour, and bucketed into four periods (late night 00-05, morning 06-11, afternoon 12-17, evening 18-23).

In [13]:
per_hour = jan.groupBy("pickup_hour").agg(F.count("*").alias("trips")).orderBy("pickup_hour")
per_hour.show(24)

period = (F.when(F.col("pickup_hour") < 6, "late night (00-05)")
           .when(F.col("pickup_hour") < 12, "morning (06-11)")
           .when(F.col("pickup_hour") < 18, "afternoon (12-17)")
           .otherwise("evening (18-23)"))

per_period = (jan.withColumn("period", period)
                 .groupBy("period").agg(F.count("*").alias("trips"))
                 .withColumn("trips_per_hour", F.round(F.col("trips") / 6))
                 .orderBy(F.desc("trips")))
per_period.show(truncate=False)

+-----------+------+
|pickup_hour| trips|
+-----------+------+
|          0|207758|
|          1|149242|
|          2|109413|
|          3| 78084|
|          4| 61423|
|          5| 75532|
|          6|178598|
|          7|304858|
|          8|373735|
|          9|365924|
|         10|361382|
|         11|375438|
|         12|401172|
|         13|404149|
|         14|433115|
|         15|452679|
|         16|420806|
|         17|468407|
|         18|515374|
|         19|475152|
|         20|423128|
|         21|409873|
|         22|369026|
|         23|281812|
+-----------+------+



+------------------+-------+--------------+
|period            |trips  |trips_per_hour|
+------------------+-------+--------------+
|afternoon (12-17) |2580328|430055.0      |
|evening (18-23)   |2474365|412394.0      |
|morning (06-11)   |1959935|326656.0      |
|late night (00-05)|681452 |113575.0      |
+------------------+-------+--------------+



**Busiest hour: 18:00–19:00 (515,374 trips). Slowest: 04:00–05:00 (61,423)** — an 8.4× spread. The daily profile is the classic city rhythm: a steep ramp between 5 and 8 am (75 k → 374 k), a plateau through the working day, an evening peak at 6–7 pm when the commute home and the going-out traffic overlap, then a slow decline to 4 am.

By period the **afternoon (12–17) is busiest with 2.58 M trips (430 k per hour)**, just ahead of the evening (2.47 M, 412 k/h); the morning carries 1.96 M and the late night 0.68 M (114 k/h). Note that the busiest *hour* falls in the evening bucket while the busiest *period* is the afternoon — bucket boundaries matter.

### 1.7 On average, which day of the week is busiest / slowest?

January 2019 starts on a Tuesday, so Tuesday, Wednesday and Thursday occur **five** times and the other days four. Summing trips per weekday would be biased; the question says *on average*, so we average the daily totals.

In [14]:
per_dow = (per_day
           .join(jan.select("pickup_dow", "pickup_dow_num").distinct(), "pickup_dow")
           .groupBy("pickup_dow_num", "pickup_dow")
           .agg(F.count("*").alias("n_days"),
                F.round(F.avg("trips")).alias("avg_trips_per_day"),
                F.sum("trips").alias("total_trips"))
           .orderBy("pickup_dow_num"))
per_dow.show()

+--------------+----------+------+-----------------+-----------+
|pickup_dow_num|pickup_dow|n_days|avg_trips_per_day|total_trips|
+--------------+----------+------+-----------------+-----------+
|             1|    Monday|     4|         226941.0|     907764|
|             2|   Tuesday|     5|         241815.0|    1209076|
|             3| Wednesday|     5|         253046.0|    1265229|
|             4|  Thursday|     5|         271398.0|    1356992|
|             5|    Friday|     4|         271788.0|    1087150|
|             6|  Saturday|     4|         252495.0|    1009979|
|             7|    Sunday|     4|         214973.0|     859890|
+--------------+----------+------+-----------------+-----------+



**Busiest weekday on average: Friday (271,788 trips/day), tied with Thursday (271,398). Slowest: Sunday (214,973), then Monday (226,941).** Demand builds up through the week, peaks Thursday–Friday, holds on Saturday and collapses on Sunday.

The `total_trips` column shows why averaging was necessary: Tuesday has the third-highest *total* only because January 2019 contains five Tuesdays, and Monday's average is dragged down by the 21 January holiday.

### 1.8 Does trip distance or the number of passengers affect the tip amount?

One thing to know about this dataset: **only credit-card tips are recorded** (cash tips are not entered by the driver), so the analysis is done on credit-card trips (`payment_type = 1`) with a real fare, and the tip is also expressed as a percentage of the fare.

In [15]:
df.groupBy("payment_type").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    F.round(100 * F.avg((F.col("tip_amount") > 0).cast("int")), 1).alias("pct_trips_with_tip"),
).orderBy("payment_type").show()

card = (jan.filter((F.col("payment_type") == 1) & (F.col("fare_amount") >= 2.5))
           .withColumn("tip_pct", 100 * F.col("tip_amount") / F.col("fare_amount")))

print("Correlation with tip_amount (credit-card trips):")
print(f"  trip_distance   : {card.stat.corr('trip_distance', 'tip_amount'):+.3f}")
print(f"  passenger_count : {card.stat.corr('passenger_count', 'tip_amount'):+.3f}")
print(f"  fare_amount     : {card.stat.corr('fare_amount', 'tip_amount'):+.3f}")

+------------+-------+-------+------------------+
|payment_type|  trips|avg_tip|pct_trips_with_tip|
+------------+-------+-------+------------------+
|           0|  28672|   0.06|               0.8|
|           1|5486263|   2.55|              95.8|
|           2|2137336|    0.0|               0.0|
|           3|  33181|    0.0|               0.1|
|           4|  11165|    0.0|               0.2|
+------------+-------+-------+------------------+

Correlation with tip_amount (credit-card trips):


  trip_distance   : +0.674


  passenger_count : +0.010


  fare_amount     : +0.186


In [16]:
tip_stats = [F.count("*").alias("trips"),
             F.round(F.avg("tip_amount"), 2).alias("avg_tip_usd"),
             F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct_of_fare"),
             F.round(100 * F.avg((F.col("tip_amount") == 0).cast("int")), 1).alias("pct_no_tip")]

print("By number of passengers")
card.groupBy("passenger_count").agg(*tip_stats).orderBy("passenger_count").show()

distance_bucket = (F.when(F.col("trip_distance") < 1, "a) < 1 mi")
                    .when(F.col("trip_distance") < 2, "b) 1-2 mi")
                    .when(F.col("trip_distance") < 5, "c) 2-5 mi")
                    .when(F.col("trip_distance") < 10, "d) 5-10 mi")
                    .when(F.col("trip_distance") < 20, "e) 10-20 mi")
                    .otherwise("f) 20+ mi"))
print("By trip distance")
card.withColumn("distance", distance_bucket).groupBy("distance").agg(*tip_stats).orderBy("distance").show()

By number of passengers


+---------------+-------+-----------+-------------------+----------+
|passenger_count|  trips|avg_tip_usd|avg_tip_pct_of_fare|pct_no_tip|
+---------------+-------+-----------+-------------------+----------+
|            0.0|  83220|       2.52|               21.9|       3.5|
|            1.0|3935762|       2.53|               21.8|       4.4|
|            2.0| 781227|       2.61|               21.8|       3.9|
|            3.0| 218502|       2.59|               21.9|       3.5|
|            4.0|  92056|        2.6|               22.1|       3.5|
|            5.0| 231249|       2.62|               22.1|       3.2|
|            6.0| 142889|       2.61|               22.1|       3.4|
|            7.0|     11|       11.3|               17.6|      18.2|
|            8.0|     27|       6.96|               14.6|      25.9|
|            9.0|      8|       3.51|               15.6|      25.0|
+---------------+-------+-----------+-------------------+----------+

By trip distance


+-----------+-------+-----------+-------------------+----------+
|   distance|  trips|avg_tip_usd|avg_tip_pct_of_fare|pct_no_tip|
+-----------+-------+-----------+-------------------+----------+
|  a) < 1 mi|1396868|       1.43|               25.1|       3.3|
|  b) 1-2 mi|1897438|       1.81|               21.9|       2.5|
|  c) 2-5 mi|1429776|       2.66|               20.0|       3.5|
| d) 5-10 mi| 433893|       4.68|               19.1|       8.9|
|e) 10-20 mi| 290196|       8.11|               18.6|      13.7|
|  f) 20+ mi|  36780|      10.53|               17.6|      16.6|
+-----------+-------+-----------+-------------------+----------+



**The payment table settles the first point:** cash trips (2.14 M) record a tip of exactly $0.00 in 100 % of cases, while 95.8 % of credit-card trips carry a tip (average $2.55). Cash tips simply are not captured, so only card trips can answer the question.

**Trip distance: yes, strongly — on the amount, not on the generosity.** The correlation between distance and tip is +0.67 on card trips, and the average tip climbs from $1.43 for trips under a mile to $10.53 beyond 20 miles. But as a *share of the fare* the tip **falls** from 25 % to 18 %, and the share of trips with no tip at all rises from 3 % to 17 %. The card reader proposes 20 / 25 / 30 % of the fare, so the tip follows the fare (which follows the distance); on long, expensive rides (airports, out of town) riders pick a lower percentage or skip it. (The fare–tip correlation of only +0.19 is an artefact: Pearson's coefficient is wrecked by the $623,259 fare of question 2.8.)

**Number of passengers: no.** Correlation +0.01; the average tip is $2.52–2.62 and 21.8–22.1 % of the fare whatever the group size from 0 to 6. The 7–9 passenger rows differ but there are 46 of them.

### 1.9 What was the highest "extra" charge, and on which trip?

In the TLC data dictionary `extra` is "miscellaneous extras and surcharges", which in 2019 should only be the **$0.50 rush-hour** and **$1.00 overnight** charges (plus $4.50 for the JFK/Newark rate).

In [17]:
(df.orderBy(F.desc("extra"))
   .select("trip_id", "tpep_pickup_datetime", "extra", "fare_amount", "total_amount",
           "trip_distance", "payment_type", "RatecodeID", "PULocationID", "DOLocationID")
   .show(3, truncate=False))

print("Most frequent values of `extra`")
df.groupBy("extra").count().orderBy(F.desc("count")).show(8)

+-----------+--------------------+------+-----------+------------+-------------+------------+----------+------------+------------+
|trip_id    |tpep_pickup_datetime|extra |fare_amount|total_amount|trip_distance|payment_type|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+------+-----------+------------+-------------+------------+----------+------------+------------+
|77314734811|2019-01-23 08:58:09 |535.38|355676.98  |356214.78   |0.0          |2           |1.0       |24          |264         |
|77316864558|2019-01-31 10:06:09 |23.04 |4.5        |28.34       |0.0          |2           |1.0       |237         |264         |
|77309954531|2019-01-03 18:32:36 |18.5  |61.0       |92.3        |16.6         |2           |3.0       |158         |1           |
+-----------+--------------------+------+-----------+------------+-------------+------------+----------+------------+------------+
only showing top 3 rows
Most frequent values of `extra`


+-----+-------+
|extra|  count|
+-----+-------+
|  0.0|4200521|
|  0.5|2116601|
|  1.0|1316575|
|  4.5|  31240|
| 2.75|  26583|
| -0.5|   2201|
|  5.5|   1377|
| -1.0|    863|
+-----+-------+
only showing top 8 rows


**Highest `extra`: $535.38, on trip `77314734811`** (23 January, 08:58, Bloomingdale → unknown, 0 miles, paid cash) — with a **fare of $355,676.98**, the second-highest of the month. The record is garbage, and so is its `extra`. The next values ($23.04 on a 0-mile trip; $18.50 on a Newark run, consistent with the $17.50 Newark surcharge plus $1) are the first ones that mean anything.

The value distribution says what the field really contains: $0 (4.2 M), the $0.50 rush-hour charge (2.1 M), the $1 overnight charge (1.3 M), $4.50 (31 k, the JFK rush-hour rate), $2.75 (26.6 k, the new congestion surcharge that some vendors already coded in this field), and negative values on reversed trips. 99.6 % of rows use one of the four expected values.

### 1.10 Strange data points / outliers

Rather than eyeballing, each suspicious condition is counted in one pass over the cached DataFrame (one `sum(condition)` per rule — Spark evaluates all of them in a single job).

In [18]:
checks = {
    "pickup outside January 2019":     ~in_jan_2019,
    "dropoff before pickup":           F.col("duration_min") < 0,
    "duration > 24 h":                 F.col("duration_min") > 24 * 60,
    "duration between 23 h and 24 h":  F.col("duration_min").between(23 * 60, 24 * 60),
    "average speed > 80 mph":          F.col("avg_mph") > 80,
    "trip_distance = 0":               F.col("trip_distance") == 0,
    "trip_distance > 100 mi":          F.col("trip_distance") > 100,
    "passenger_count = 0":             F.col("passenger_count") == 0,
    "passenger_count > 6":             F.col("passenger_count") > 6,
    "fare_amount <= 0":                F.col("fare_amount") <= 0,
    "fare_amount > 500":               F.col("fare_amount") > 500,
    "total_amount < 0":                F.col("total_amount") < 0,
    "RatecodeID not in 1..6":          ~F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6),
    "extra not in {0, 0.5, 1, 4.5}":   ~F.col("extra").isin(0.0, 0.5, 1.0, 4.5),
    "tip larger than the fare":        F.col("tip_amount") > F.col("fare_amount"),
    "fare > 0 but distance = 0 and duration < 1 min":
        (F.col("fare_amount") > 0) & (F.col("trip_distance") == 0) & (F.col("duration_min") < 1),
}

outliers = (df.agg(*[F.sum(cond.cast("int")).alias(name) for name, cond in checks.items()])
              .toPandas().T.rename(columns={0: "rows"}))
outliers["pct_of_file"] = (100 * outliers["rows"] / n_rows).round(3)
outliers

,rows,pct_of_file
pickup outside January 2019,537,0.007
dropoff before pickup,4,0.000
duration > 24 h,5,0.000
duration between 23 h and 24 h,18739,0.243
average speed > 80 mph,6147,0.080
trip_distance = 0,55089,0.716
trip_distance > 100 mi,32,0.000
passenger_count = 0,117381,1.525
passenger_count > 6,57,0.001
fare_amount <= 0,9770,0.127


In [19]:
print("Negative fares by payment type (3 = no charge, 4 = dispute, 6 = voided trip)")
df.filter("fare_amount < 0").groupBy("payment_type").count().orderBy("payment_type").show()

print("A few of the > 100-mile trips")
(df.filter("trip_distance > 100")
   .select("trip_id", "tpep_pickup_datetime", "trip_distance", "duration_min", "fare_amount", "RatecodeID", "PULocationID", "DOLocationID")
   .orderBy(F.desc("trip_distance")).show(5))

Negative fares by payment type (3 = no charge, 4 = dispute, 6 = voided trip)


+------------+-----+
|payment_type|count|
+------------+-----+
|           0|    2|
|           1|    1|
|           2|  376|
|           3| 4083|
|           4| 2667|
+------------+-----+

A few of the > 100-mile trips
+-----------+--------------------+-------------+------------------+-----------+----------+------------+------------+
|    trip_id|tpep_pickup_datetime|trip_distance|      duration_min|fare_amount|RatecodeID|PULocationID|DOLocationID|
+-----------+--------------------+-------------+------------------+-----------+----------+------------+------------+
|77315485419| 2019-01-25 21:56:39|        831.8| 9.483333333333333|        8.5|       1.0|         140|         239|
|77313697961| 2019-01-18 16:32:24|        700.7| 6.933333333333334|        6.0|       1.0|         236|         262|
|77316182313| 2019-01-28 17:24:11|       214.01| 673.0833333333334|      760.0|       1.0|         265|         265|
|77314118862| 2019-01-20 12:22:24|       211.36|             283.2|       50.0

**Reasoning.** Each rule targets a value that is physically or commercially impossible, and the table gives its weight. Ordered by what they reveal:

- **Fields the driver did not fill in** — 117,381 trips with 0 passengers (1.5 %). Harmless for most questions, must be excluded from the passenger average.
- **Meter started and cancelled** — 55,089 zero-distance trips (0.7 %), of which 41,105 also last under a minute *and* carry a fare: the minimum $2.50 flag-drop billed on a ride that never happened, or a manual entry.
- **Reversed transactions** — 9,770 non-positive fares (0.13 %) and 7,127 negative totals. The second table shows they belong almost entirely to payment types 3 ("no charge") and 4 ("dispute"): they are **refunds recorded as negative trips**, not rides. They should be removed before any revenue sum.
- **Sensor / clock errors** — 32 trips over 100 miles (831.8 miles in 9 minutes) and 6,147 with an average speed above 80 mph, 5 longer than 24 h, 4 with a negative duration, 18,739 lasting between 23 and 24 h (drop-off stamped one day late), 537 pickups outside the month (2001 to 2088).
- **Keying errors** — 34 fares above $500, topped by **$623,259.86** and $355,676.98 (a decimal point lost somewhere); 57 rides with 7–9 passengers; 252 records with rate code 99 ("unknown"); 13,784 tips larger than the fare (mostly tiny fares with a fixed-amount tip).
- **Field misuse** — 31,680 `extra` values outside the four legal amounts.

None of these categories exceeds 2 % of the file, but they own every maximum and minimum. That is why Part 2 reports **medians next to means**, and why "longest / highest" questions are answered twice: raw, then on plausible trips.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### 2.1 Load the taxi zone lookup table

The lookup is a small CSV (265 zones). Following the note above on data sources, the schema is given **explicitly** rather than inferred.

In [20]:
zone_file = download("misc/taxi_zone_lookup.csv")

zone_schema = T.StructType([
    T.StructField("LocationID", T.IntegerType()),
    T.StructField("Borough", T.StringType()),
    T.StructField("Zone", T.StringType()),
    T.StructField("service_zone", T.StringType()),
])
zones = (spark.read.format("csv")
              .option("header", "true")
              .schema(zone_schema)
              .load(str(zone_file)))

print(f"{zones.count()} zones")
zones.show(5, truncate=False)
zones.groupBy("Borough").agg(F.count("*").alias("zones")).orderBy(F.desc("zones")).show()

265 zones
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows


+-------------+-----+
|      Borough|zones|
+-------------+-----+
|       Queens|   69|
|    Manhattan|   69|
|     Brooklyn|   61|
|        Bronx|   43|
|Staten Island|   20|
|          EWR|    1|
|      Unknown|    1|
|          N/A|    1|
+-------------+-----+



### 2.2 Enrich the trips with pickup and drop-off boroughs (join)

The trips table only has `PULocationID` / `DOLocationID`. Joining the lookup **twice** (once per end of the trip) gives the pickup and drop-off borough and zone. The lookup is tiny, so `F.broadcast()` makes it a broadcast-hash join: no shuffle of the 7.7 M trip rows. A **left** join keeps trips whose location id is not in the lookup, so we can count them.

In [21]:
pu = zones.select(F.col("LocationID").alias("PULocationID"),
                  F.col("Borough").alias("PU_Borough"), F.col("Zone").alias("PU_Zone"))
do = zones.select(F.col("LocationID").alias("DOLocationID"),
                  F.col("Borough").alias("DO_Borough"), F.col("Zone").alias("DO_Zone"))

jan_b = (jan.join(F.broadcast(pu), "PULocationID", "left")
            .join(F.broadcast(do), "DOLocationID", "left"))

jan_b.select("trip_id", "PULocationID", "PU_Borough", "PU_Zone", "DOLocationID", "DO_Borough", "DO_Zone").show(5, truncate=False)
print(f"trips whose pickup location is not in the lookup:  {jan_b.filter(F.col('PU_Borough').isNull()).count():,}")
print(f"trips whose dropoff location is not in the lookup: {jan_b.filter(F.col('DO_Borough').isNull()).count():,}")

+-----------+------------+----------+-----------------------------+------------+----------+-----------------------------+
|trip_id    |PULocationID|PU_Borough|PU_Zone                      |DOLocationID|DO_Borough|DO_Zone                      |
+-----------+------------+----------+-----------------------------+------------+----------+-----------------------------+
|77309411328|151         |Manhattan |Manhattan Valley             |239         |Manhattan |Upper West Side South        |
|77309411329|239         |Manhattan |Upper West Side South        |246         |Manhattan |West Chelsea/Hudson Yards    |
|77309411335|163         |Manhattan |Midtown North                |229         |Manhattan |Sutton Place/Turtle Bay North|
|77309411336|229         |Manhattan |Sutton Place/Turtle Bay North|7           |Queens    |Astoria                      |
|77309411337|141         |Manhattan |Lenox Hill West              |234         |Manhattan |Union Sq                     |
+-----------+-----------

trips whose pickup location is not in the lookup:  0


trips whose dropoff location is not in the lookup: 0


### 2.3 Which borough had the most pickups? The most drop-offs?

In [22]:
n_jan = jan_b.count()

pickups  = jan_b.groupBy(F.col("PU_Borough").alias("borough")).agg(F.count("*").alias("pickups"))
dropoffs = jan_b.groupBy(F.col("DO_Borough").alias("borough")).agg(F.count("*").alias("dropoffs"))

by_borough = (pickups.join(dropoffs, "borough", "full").fillna(0)
                     .withColumn("pickup_share_pct", F.round(100 * F.col("pickups") / n_jan, 2))
                     .withColumn("dropoff_share_pct", F.round(100 * F.col("dropoffs") / n_jan, 2))
                     .withColumn("net_inflow", F.col("dropoffs") - F.col("pickups"))
                     .orderBy(F.desc("pickups")))
by_borough.show()

+-------------+-------+--------+----------------+-----------------+----------+
|      borough|pickups|dropoffs|pickup_share_pct|dropoff_share_pct|net_inflow|
+-------------+-------+--------+----------------+-----------------+----------+
|    Manhattan|6950511| 6816936|           90.31|            88.58|   -133575|
|       Queens| 471113|  340914|            6.12|             4.43|   -130199|
|      Unknown| 159807|  149091|            2.08|             1.94|    -10716|
|     Brooklyn|  91896|  301074|            1.19|             3.91|    209178|
|        Bronx|  18056|   58068|            0.23|             0.75|     40012|
|          N/A|   3890|   16900|            0.05|             0.22|     13010|
|          EWR|    446|   10913|            0.01|             0.14|     10467|
|Staten Island|    361|    2184|             0.0|             0.03|      1823|
+-------------+-------+--------+----------------+-----------------+----------+



**Manhattan, by an enormous margin: 6.95 M pickups (90.3 %) and 6.82 M drop-offs (88.6 %).** Queens is second for pickups (471 k, 6.1 %) — essentially JFK and LaGuardia — but only third for drop-offs (4.4 %).

The `net_inflow` column is the interesting one: **Brooklyn receives 3.3× more yellow-cab drop-offs than it generates pickups** (+209 k), the Bronx 3.2× (+40 k), Newark Airport 24× (+10 k). Yellow cabs carry people *out* of Manhattan and the airports into the other boroughs, but almost never pick anyone up there — that is the market the green "Boro taxis" were created for (see 4.3). "Unknown" (2.1 %) is zone 264/265, i.e. the meter had no location fix.

### 2.4 Busy and slow times by borough

For each pickup borough: the busiest and slowest hour (window function ranking hours within each borough), then the four time periods as a pivot table.

In [23]:
borough_hour = jan_b.groupBy("PU_Borough", "pickup_hour").agg(F.count("*").alias("trips"))

w_busy = Window.partitionBy("PU_Borough").orderBy(F.desc("trips"))
w_slow = Window.partitionBy("PU_Borough").orderBy(F.asc("trips"))

busiest = (borough_hour.withColumn("rk", F.row_number().over(w_busy)).filter("rk = 1")
                       .select("PU_Borough", F.col("pickup_hour").alias("busiest_hour"), F.col("trips").alias("trips_at_busiest")))
slowest = (borough_hour.withColumn("rk", F.row_number().over(w_slow)).filter("rk = 1")
                       .select("PU_Borough", F.col("pickup_hour").alias("slowest_hour"), F.col("trips").alias("trips_at_slowest")))

busiest.join(slowest, "PU_Borough").orderBy(F.desc("trips_at_busiest")).show()

print("Pickups per period of the day, by borough (share of the borough's trips, %)")
period_pivot = (jan_b.withColumn("period", period)
                     .groupBy("PU_Borough").pivot("period").count().fillna(0))
period_cols = [c for c in period_pivot.columns if c != "PU_Borough"]
total = sum(F.col(c) for c in period_cols)
(period_pivot.select("PU_Borough", *[F.round(100 * F.col(c) / total, 1).alias(c) for c in period_cols])
             .orderBy("PU_Borough").show(truncate=False))

+-------------+------------+----------------+------------+----------------+
|   PU_Borough|busiest_hour|trips_at_busiest|slowest_hour|trips_at_slowest|
+-------------+------------+----------------+------------+----------------+
|    Manhattan|          18|          471524|           4|           53446|
|       Queens|          16|           29880|           3|            3084|
|      Unknown|          18|           10751|           4|            1465|
|     Brooklyn|           8|            6935|           3|            1919|
|        Bronx|           7|            1803|           3|             225|
|          N/A|          19|             214|           6|              88|
|          EWR|          15|              54|          23|               1|
|Staten Island|           8|              36|           1|               3|
+-------------+------------+----------------+------------+----------------+

Pickups per period of the day, by borough (share of the borough's trips, %)


+-------------+-----------------+---------------+------------------+---------------+
|PU_Borough   |afternoon (12-17)|evening (18-23)|late night (00-05)|morning (06-11)|
+-------------+-----------------+---------------+------------------+---------------+
|Bronx        |29.6             |15.9           |12.0              |42.5           |
|Brooklyn     |24.7             |26.1           |17.2              |32.0           |
|EWR          |55.6             |13.5           |6.7               |24.2           |
|Manhattan    |33.6             |32.1           |8.7               |25.5           |
|N/A          |27.2             |30.0           |19.5              |23.3           |
|Queens       |33.4             |34.6           |9.1               |22.9           |
|Staten Island|31.0             |14.4           |13.0              |41.6           |
|Unknown      |34.3             |31.9           |8.8               |24.9           |
+-------------+-----------------+---------------+----------------

- **Manhattan** peaks at 18:00 (471,524 trips in the month) and bottoms at 04:00, an 8.8× spread; afternoon and evening each carry a third of its trips.
- **Queens** peaks at 16:00 and is evening-heavy (34.6 %): afternoon flight arrivals at JFK/LaGuardia.
- **Brooklyn and the Bronx are morning boroughs**: peak at 08:00 and 07:00, with 32 % and 42.5 % of their pickups between 6 and 11 am. Yellow cabs there are mostly a morning commute into Manhattan; the evening return trips show up as *drop-offs* in 2.3, not as pickups.
- **Late night** matters most in Brooklyn (17 %) and "N/A" (19.5 %): nightlife rides back home.
- Staten Island (361 trips), EWR (446) and N/A (3,890) are too small for their hourly extremes to mean much — Newark's "slowest hour" is a single trip.

### 2.5 Busiest day of the week by borough

Same idea as 1.7 (average of the daily totals, because some weekdays occur five times in January 2019), then a ranking within each borough.

In [24]:
borough_dow = (jan_b.groupBy("PU_Borough", "pickup_date", "pickup_dow").agg(F.count("*").alias("trips"))
                    .groupBy("PU_Borough", "pickup_dow").agg(F.round(F.avg("trips")).alias("avg_trips_per_day")))

w_dow = Window.partitionBy("PU_Borough").orderBy(F.desc("avg_trips_per_day"))
ranked = borough_dow.withColumn("rank", F.row_number().over(w_dow))

print("Busiest weekday per borough")
(ranked.filter("rank = 1").select("PU_Borough", F.col("pickup_dow").alias("busiest_weekday"), "avg_trips_per_day")
       .orderBy(F.desc("avg_trips_per_day")).show())

print("Slowest weekday per borough")
(ranked.filter("rank = 7").select("PU_Borough", F.col("pickup_dow").alias("slowest_weekday"), "avg_trips_per_day")
       .orderBy(F.desc("avg_trips_per_day")).show())

print("Full profile (average trips per day)")
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
(borough_dow.groupBy("PU_Borough").pivot("pickup_dow", dow_order).sum("avg_trips_per_day")
            .orderBy("PU_Borough").show())

Busiest weekday per borough


+-------------+---------------+-----------------+
|   PU_Borough|busiest_weekday|avg_trips_per_day|
+-------------+---------------+-----------------+
|    Manhattan|         Friday|         246224.0|
|       Queens|         Monday|          16662.0|
|      Unknown|       Thursday|           5785.0|
|     Brooklyn|         Friday|           3273.0|
|        Bronx|         Friday|            667.0|
|          N/A|        Tuesday|            141.0|
|          EWR|         Friday|             19.0|
|Staten Island|         Friday|             16.0|
+-------------+---------------+-----------------+

Slowest weekday per borough


+-------------+---------------+-----------------+
|   PU_Borough|slowest_weekday|avg_trips_per_day|
+-------------+---------------+-----------------+
|    Manhattan|         Sunday|         192553.0|
|       Queens|       Saturday|          11915.0|
|      Unknown|         Sunday|           4174.0|
|     Brooklyn|         Monday|           2377.0|
|        Bronx|       Saturday|            482.0|
|          N/A|         Friday|            111.0|
|Staten Island|      Wednesday|             10.0|
|          EWR|         Monday|              8.0|
+-------------+---------------+-----------------+

Full profile (average trips per day)


+-------------+--------+--------+---------+--------+--------+--------+--------+
|   PU_Borough|  Monday| Tuesday|Wednesday|Thursday|  Friday|Saturday|  Sunday|
+-------------+--------+--------+---------+--------+--------+--------+--------+
|        Bronx|   543.0|   612.0|    600.0|   624.0|   667.0|   482.0|   528.0|
|     Brooklyn|  2377.0|  3156.0|   3020.0|  3143.0|  3273.0|  2901.0|  2775.0|
|          EWR|     8.0|    15.0|     17.0|    12.0|    19.0|    14.0|    17.0|
|    Manhattan|201858.0|217239.0| 228953.0|245903.0|246224.0|231875.0|192553.0|
|          N/A|   129.0|   141.0|    127.0|   127.0|   111.0|   123.0|   117.0|
|       Queens| 16662.0| 15737.0|  15163.0| 15792.0| 16038.0| 11915.0| 14798.0|
|Staten Island|    11.0|    11.0|     10.0|    12.0|    16.0|    10.0|    11.0|
|      Unknown|  5354.0|  4905.0|   5156.0|  5785.0|  5441.0|  5176.0|  4174.0|
+-------------+--------+--------+---------+--------+--------+--------+--------+



**Friday is the busiest weekday for Manhattan (246 k pickups/day), Brooklyn, the Bronx, Staten Island and Newark. Queens is the exception: busiest on Monday (16.7 k), slowest on Saturday (11.9 k)** — airport traffic follows business travel, with Monday-morning arrivals and a weekend lull, then a Sunday rebound (14.8 k) as travellers come back. Manhattan's slowest day is Sunday (193 k), and its full profile climbs steadily from Monday (202 k) to Friday.

### 2.6 Average trip distance by borough

In [25]:
dist_by_borough = (jan_b.groupBy("PU_Borough").agg(
        F.count("*").alias("trips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.percentile_approx("trip_distance", 0.5), 2).alias("median_distance_mi"),
        F.round(F.avg("duration_min"), 1).alias("avg_duration_min"))
    .orderBy(F.desc("trips")))
dist_by_borough.show()

+-------------+-------+---------------+------------------+----------------+
|   PU_Borough|  trips|avg_distance_mi|median_distance_mi|avg_duration_min|
+-------------+-------+---------------+------------------+----------------+
|    Manhattan|6950511|           2.23|              1.46|            15.3|
|       Queens| 471113|          11.28|              10.6|            33.7|
|      Unknown| 159807|           2.42|               1.4|            14.0|
|     Brooklyn|  91896|           4.79|              3.14|            21.6|
|        Bronx|  18056|           7.23|              5.57|            25.6|
|          N/A|   3890|           3.19|              0.32|            13.8|
|          EWR|    446|           2.64|               0.0|            11.3|
|Staten Island|    361|           12.5|             11.66|            33.9|
+-------------+-------+---------------+------------------+----------------+



**Average distance by pickup borough: Manhattan 2.2 miles (median 1.5, 15 minutes); Brooklyn 4.8; Bronx 7.2; Queens 11.3 (median 10.6, 34 minutes); Staten Island 12.5.** Manhattan rides are short intra-borough hops; Queens rides are airport runs into Manhattan, ten miles long and half an hour. The mean exceeds the median everywhere — the distribution is right-skewed and the outliers of 1.10 pull the mean up (Manhattan's mean is 50 % above its median). Newark's median of 0.0 miles is another data oddity: half its 446 "pickups" have no distance at all.

### 2.7 Average trip fare by borough

In [26]:
fare_by_borough = (jan_b.groupBy("PU_Borough").agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.percentile_approx("fare_amount", 0.5), 2).alias("median_fare"),
        F.round(F.avg("total_amount"), 2).alias("avg_total"),
        F.round(F.avg("fare_amount") / F.avg("trip_distance"), 2).alias("avg_fare_per_mile"))
    .orderBy(F.desc("trips")))
fare_by_borough.show()

+-------------+-------+--------+-----------+---------+-----------------+
|   PU_Borough|  trips|avg_fare|median_fare|avg_total|avg_fare_per_mile|
+-------------+-------+--------+-----------+---------+-----------------+
|    Manhattan|6950511|   10.79|        8.5|    13.67|             4.84|
|       Queens| 471113|   35.14|       34.0|    44.45|             3.11|
|      Unknown| 159807|   14.94|        8.5|    18.17|             6.19|
|     Brooklyn|  91896|   18.65|       14.0|    21.62|              3.9|
|        Bronx|  18056|   26.27|       22.0|    29.31|             3.63|
|          N/A|   3890|   59.57|       43.0|    69.55|            18.65|
|          EWR|    446|   76.24|       85.0|    92.78|            28.87|
|Staten Island|    361|   45.29|       40.5|    53.59|             3.62|
+-------------+-------+--------+-----------+---------+-----------------+



**Average fare by pickup borough: Manhattan $10.79 (median $8.50); Brooklyn $18.65; Bronx $26.27; Queens $35.14 (median $34, close to the $52 JFK flat rate once tolls and surcharges are separated out); Staten Island $45; outside NYC $60; Newark $76.** Fares track distance, but not proportionally: the **fare per mile is highest in Manhattan ($4.84)** and lowest in Queens ($3.11) because the meter also bills waiting time, and Manhattan traffic is slow. The gap between Manhattan's mean and median is again the outliers: a single $623,259.86 record shifts the mean of 6.95 M trips by 9 cents.

### 2.8 Highest and lowest fare amounts, and the boroughs associated with them

In [27]:
fare_cols = ["trip_id", "tpep_pickup_datetime", "fare_amount", "total_amount", "trip_distance", "duration_min", "avg_mph",
             "payment_type", "RatecodeID", "PU_Borough", "PU_Zone", "DO_Borough", "DO_Zone"]

print("Highest fares"); jan_b.orderBy(F.desc("fare_amount")).select(*fare_cols).show(3, truncate=False)
print("Lowest fares");  jan_b.orderBy("fare_amount").select(*fare_cols).show(3, truncate=False)

print("Highest *plausible* fare (positive distance, 1 min - 24 h, <= 80 mph, standard or JFK rate, $1-20 per mile)")
fare_per_mile = F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance"))
(jan_b.filter((F.col("trip_distance") > 0) & (F.col("duration_min").between(1, 24 * 60)) & (F.col("avg_mph") <= 80)
              & (F.col("RatecodeID").isin(1, 2)) & fare_per_mile.between(1, 20))
      .orderBy(F.desc("fare_amount")).select(*fare_cols).show(3, truncate=False))

Highest fares


+-----------+--------------------+-----------+------------+-------------+------------+-----------------+------------+----------+----------+---------------------+----------+--------+
|trip_id    |tpep_pickup_datetime|fare_amount|total_amount|trip_distance|duration_min|avg_mph          |payment_type|RatecodeID|PU_Borough|PU_Zone              |DO_Borough|DO_Zone |
+-----------+--------------------+-----------+------------+-------------+------------+-----------------+------------+----------+----------+---------------------+----------+--------+
|77311910983|2019-01-11 19:33:15 |623259.86  |623261.66   |2.4          |19.9        |7.236180904522612|3           |1.0       |Manhattan |Upper East Side South|Manhattan |Flatiron|
|77314734811|2019-01-23 08:58:09 |355676.98  |356214.78   |0.0          |0.0         |NULL             |2           |1.0       |Manhattan |Bloomingdale         |Unknown   |N/A     |
|77311571299|2019-01-10 16:08:10 |36090.3    |36090.3     |0.0          |0.0         |NULL

+-----------+--------------------+-----------+------------+-------------+-------------------+-------+------------+----------+----------+---------------------------------+----------+---------------------------------+
|trip_id    |tpep_pickup_datetime|fare_amount|total_amount|trip_distance|duration_min       |avg_mph|payment_type|RatecodeID|PU_Borough|PU_Zone                          |DO_Borough|DO_Zone                          |
+-----------+--------------------+-----------+------------+-------------+-------------------+-------+------------+----------+----------+---------------------------------+----------+---------------------------------+
|77314301977|2019-01-21 12:01:45 |-362.0     |-362.8      |0.0          |6.85               |0.0    |4           |5.0       |Queens    |JFK Airport                      |Queens    |JFK Airport                      |
|77315719528|2019-01-26 18:46:42 |-320.0     |-320.3      |0.0          |0.13333333333333333|0.0    |3           |5.0       |N/A       |

+-----------+--------------------+-----------+------------+-------------+------------------+------------------+------------+----------+----------+-----------------+----------+--------------+
|trip_id    |tpep_pickup_datetime|fare_amount|total_amount|trip_distance|duration_min      |avg_mph           |payment_type|RatecodeID|PU_Borough|PU_Zone          |DO_Borough|DO_Zone       |
+-----------+--------------------+-----------+------------+-------------+------------------+------------------+------------+----------+----------+-----------------+----------+--------------+
|77316182313|2019-01-28 17:24:11 |760.0      |761.8       |214.01       |673.0833333333334 |19.077281168750773|2           |1.0       |N/A       |Outside of NYC   |N/A       |Outside of NYC|
|77314814037|2019-01-23 14:24:49 |322.0      |396.22      |68.62        |146.7             |28.065439672801638|1           |1.0       |Manhattan |East Harlem North|N/A       |Outside of NYC|
|77309467806|2019-01-01 04:17:02 |306.5      

**Highest fare: $623,259.86**, trip `77311910983`, 11 January 19:33, **Manhattan** (Upper East Side South → Flatiron), 2.4 miles in 20 minutes, payment type 3 "no charge" — nobody paid it, it is a keying error on an otherwise ordinary $15 ride. Next come $355,676.98 (Manhattan, Bloomingdale, 0 miles) and $36,090.30 (unknown zone, rate code 99).

**Lowest fare: −$362.00**, trip `77314301977`, 21 January, **Queens** (JFK Airport → JFK Airport), rate code 5 (negotiated), payment type 4 "dispute"; then −$320 outside NYC and −$300 in the Bronx, all negotiated fares reversed through a dispute or no-charge record. These are refunds, not trips.

Filtering on plausibility (positive distance, 1 min – 24 h, ≤ 80 mph, standard/JFK rate, $1–20 per mile) gives the **highest real fare: $760 for the 214-mile, 11-hour ride from and to an unknown zone**, then $322 for 69 miles (Manhattan → outside NYC) and $306.50 for 120 miles at 4 am on New Year's Day. Note that the naive filter of 1.4 lets the $623,259 record through — its distance and duration are perfectly normal — so sanity checks must also look at the value itself.

### 2.9 The most recent January available: has anything changed?

The instructions mention January 2025; at the time of writing (September 2026) the most recent January published by the TLC is **January 2026**, so both are loaded and compared with 2019. The three files are read separately and reduced to one row of metrics each, then stacked with `unionByName` — this sidesteps the schema differences between years shown first.

In [28]:
jan_2025_trip_data = download("trip-data/yellow_tripdata_2025-01.parquet")
jan_2026_trip_data = download("trip-data/yellow_tripdata_2026-01.parquet")
df_2025 = spark.read.parquet(str(jan_2025_trip_data))
df_2026 = spark.read.parquet(str(jan_2026_trip_data))


def schema_dict(d):
    return {f.name: f.dataType.simpleString() for f in d.schema.fields}


s2019, s2026 = schema_dict(df_trips), schema_dict(df_2026)
print("columns only in 2026 :", sorted(set(s2026) - set(s2019)))
print("columns only in 2019 :", sorted(set(s2019) - set(s2026)))
print("type changes         :", {c: f"{s2019[c]} -> {s2026[c]}" for c in s2019 if c in s2026 and s2019[c] != s2026[c]})

columns only in 2026 : ['Airport_fee', 'cbd_congestion_fee']
columns only in 2019 : ['airport_fee']
type changes         : {'VendorID': 'bigint -> int', 'passenger_count': 'double -> bigint', 'RatecodeID': 'double -> bigint', 'PULocationID': 'bigint -> int', 'DOLocationID': 'bigint -> int'}


In [29]:
def january_metrics(d, year: int):
    """One row of averages for the pickups that really fall in January of `year`."""
    d = (d.withColumn("passenger_count", F.col("passenger_count").cast("double"))
          .withColumn("duration_min",
                      F.timestamp_diff("SECOND", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")) / 60.0)
          .filter((F.col("tpep_pickup_datetime") >= f"{year}-01-01") & (F.col("tpep_pickup_datetime") < f"{year}-02-01")))
    card = F.col("payment_type") == 1
    return d.agg(
        F.lit(year).alias("january"),
        F.count("*").alias("trips"),
        F.round(F.count("*") / 31).alias("trips_per_day"),
        F.round(F.avg("passenger_count"), 3).alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
        F.round(F.avg(F.when(F.col("trip_distance") < 100, F.col("trip_distance"))), 2).alias("avg_distance_lt100mi"),
        F.round(F.percentile_approx("trip_distance", 0.5), 2).alias("median_distance_mi"),
        F.round(F.avg("duration_min"), 1).alias("avg_duration_min"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.percentile_approx("fare_amount", 0.5), 2).alias("median_fare"),
        F.round(F.avg("total_amount"), 2).alias("avg_total"),
        F.round(F.avg(F.when(card, F.col("tip_amount"))), 2).alias("avg_tip_card"),
        F.round(100 * F.avg(card.cast("int")), 1).alias("card_share_pct"),
        F.round(100 * F.avg((F.col("payment_type") == 0).cast("int")), 1).alias("unknown_payment_pct"),
    )


comparison = (january_metrics(df_trips, 2019)
              .unionByName(january_metrics(df_2025, 2025))
              .unionByName(january_metrics(df_2026, 2026)))
comparison.show()

+-------+-------+-------------+--------------+---------------+--------------------+------------------+----------------+--------+-----------+---------+------------+--------------+-------------------+
|january|  trips|trips_per_day|avg_passengers|avg_distance_mi|avg_distance_lt100mi|median_distance_mi|avg_duration_min|avg_fare|median_fare|avg_total|avg_tip_card|card_share_pct|unknown_payment_pct|
+-------+-------+-------------+--------------+---------------+--------------------+------------------+----------------+--------+-----------+---------+------------+--------------+-------------------+
|   2019|7696080|     248261.0|         1.567|           2.83|                2.83|              1.54|            16.5|   12.53|        9.0|    15.81|        2.55|          71.3|                0.4|
|   2025|3475204|     112103.0|         1.298|           5.86|                3.09|              1.67|            15.0|   17.08|      12.12|    25.61|        4.11|          70.3|               15.5|
|   2

**Schema first.** The 2025 and 2026 files add `Airport_fee` (capital A — the 2019 file has an all-null lowercase `airport_fee`, an artefact of the TLC's CSV-to-parquet conversion) and `cbd_congestion_fee`, the Manhattan congestion-pricing charge introduced on 5 January 2025; identifiers and `passenger_count` become integers. `unionByName` on one-row aggregates avoids fighting these differences.

**Yes, almost every average moved:**

- **Volume: 7.70 M trips in January 2019 → 3.48 M in 2025 → 3.72 M in 2026** (−52 % vs 2019, +7 % 2026 vs 2025). Ride-hailing and the pandemic halved the yellow-cab market; it is slowly growing again.
- **Passengers per trip: 1.57 → 1.30 → 1.26.** More solo rides — with the caveat that 1.09 M 2026 records (29 %) have no passenger count *and* an unknown payment type, so the 2026 figures describe the 71 % of trips that have a full record.
- **Distance: the median goes 1.54 → 1.67 → 1.81 miles (+18 %), trips are getting longer.** The raw mean (2.83 → 5.86 → 6.46) says something different because the 2025 and 2026 files each contain 162 trips of 100 to 276,423 "miles"; the mean of trips under 100 miles (2.83 → 3.09 → 3.37) agrees with the median. Same lesson as 1.10, on a different year.
- **Price: average fare $12.53 → $17.08 → $20.80 (+66 %), average total $15.81 → $25.61 → $29.18 (+85 %).** The December 2022 meter increase (+23 %), then the congestion fee and further surcharges, and fewer but longer rides.
- **Tips: $2.55 → $4.11 → $4.14** on card trips, i.e. stable at about 20 % of a higher fare. Trip duration barely moves (16.5 → 15.0 → 17.2 minutes).
- The **card share (71 % → 70 % → 60 %)** should not be read as a return of cash: cash falls from 28 % to 8 %, and the drop comes from the 29 % of 2026 records with payment type 0 "unknown".

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

The main notebook used the DataFrame API, so the three questions below are re-answered in **pure Spark SQL**. Two of them involve a join with the zone lookup. Each SQL result is compared **programmatically** with the DataFrame result from Part 1/2 (`assert`), so the two APIs are shown to be two syntaxes for the same Catalyst plan.

The `trips` view is the January 2019 DataFrame **without** the borough columns, so the joins really have to be written in SQL.

In [30]:
jan.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |    trips|       true|
|         |    zones|       true|
+---------+---------+-----------+



### 3.1 (SQL, with a join) Which borough had the most pickups? — re-answers 2.3

In [31]:
sql_pickups = spark.sql("""
    SELECT z.Borough                                                     AS borough,
           COUNT(*)                                                      AS pickups,
           ROUND(100 * COUNT(*) / (SELECT COUNT(*) FROM trips), 2)       AS pickup_share_pct
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY pickups DESC
""")
sql_pickups.show()

expected = [(r.borough, r.pickups) for r in by_borough.select("borough", "pickups").collect()]
got      = [(r.borough, r.pickups) for r in sql_pickups.collect()]
assert got == expected, (got, expected)
print("SQL result identical to the DataFrame result of 2.3")

+-------------+-------+----------------+
|      borough|pickups|pickup_share_pct|
+-------------+-------+----------------+
|    Manhattan|6950511|           90.31|
|       Queens| 471113|            6.12|
|      Unknown| 159807|            2.08|
|     Brooklyn|  91896|            1.19|
|        Bronx|  18056|            0.23|
|          N/A|   3890|            0.05|
|          EWR|    446|            0.01|
|Staten Island|    361|             0.0|
+-------------+-------+----------------+



SQL result identical to the DataFrame result of 2.3


### 3.2 (SQL) Busiest and slowest hour of the day — re-answers 1.6

In [32]:
sql_hours = spark.sql("""
    WITH per_hour AS (
        SELECT hour(tpep_pickup_datetime) AS pickup_hour, COUNT(*) AS trips
        FROM trips
        GROUP BY hour(tpep_pickup_datetime)
    )
    SELECT pickup_hour, trips,
           CASE WHEN trips = (SELECT MAX(trips) FROM per_hour) THEN 'busiest'
                WHEN trips = (SELECT MIN(trips) FROM per_hour) THEN 'slowest' END AS extreme
    FROM per_hour
    WHERE trips IN ((SELECT MAX(trips) FROM per_hour), (SELECT MIN(trips) FROM per_hour))
    ORDER BY trips DESC
""")
sql_hours.show()

df_hours = per_hour.orderBy(F.desc("trips")).collect()
expected = [(df_hours[0].pickup_hour, df_hours[0].trips), (df_hours[-1].pickup_hour, df_hours[-1].trips)]
got = [(r.pickup_hour, r.trips) for r in sql_hours.collect()]
assert got == expected, (got, expected)
print("SQL result identical to the DataFrame result of 1.6")

+-----------+------+-------+
|pickup_hour| trips|extreme|
+-----------+------+-------+
|         18|515374|busiest|
|          4| 61423|slowest|
+-----------+------+-------+



SQL result identical to the DataFrame result of 1.6


### 3.3 (SQL, with a double join) Average, median, highest and lowest fare by pickup borough — re-answers 2.7 and 2.8

In [33]:
sql_fares = spark.sql("""
    WITH enriched AS (
        SELECT t.*,
               pu.Borough AS pu_borough,
               d.Borough  AS do_borough
        FROM trips t
        LEFT JOIN zones pu ON t.PULocationID = pu.LocationID
        LEFT JOIN zones d  ON t.DOLocationID = d.LocationID
    )
    SELECT pu_borough,
           COUNT(*)                                     AS trips,
           ROUND(AVG(fare_amount), 2)                   AS avg_fare,
           ROUND(percentile_approx(fare_amount, 0.5), 2) AS median_fare,
           MAX(fare_amount)                             AS highest_fare,
           MIN(fare_amount)                             AS lowest_fare
    FROM enriched
    GROUP BY pu_borough
    ORDER BY trips DESC
""")
sql_fares.show()

expected = [(r.PU_Borough, r.trips, r.avg_fare) for r in fare_by_borough.collect()]
got      = [(r.pu_borough, r.trips, r.avg_fare) for r in sql_fares.collect()]
assert got == expected, (got, expected)
print("SQL result identical to the DataFrame result of 2.7")

+-------------+-------+--------+-----------+------------+-----------+
|   pu_borough|  trips|avg_fare|median_fare|highest_fare|lowest_fare|
+-------------+-------+--------+-----------+------------+-----------+
|    Manhattan|6950511|   10.79|        8.5|   623259.86|     -252.0|
|       Queens| 471113|   35.14|       34.0|      655.35|     -362.0|
|      Unknown| 159807|   14.94|        8.5|     36090.3|     -224.0|
|     Brooklyn|  91896|   18.65|       14.0|       412.0|     -165.0|
|        Bronx|  18056|   26.27|       22.0|       679.5|     -300.0|
|          N/A|   3890|   59.57|       43.0|     1079.15|     -320.0|
|          EWR|    446|   76.24|       85.0|       342.0|    -142.06|
|Staten Island|    361|   45.29|       40.5|      355.55|      -12.0|
+-------------+-------+--------+-----------+------------+-----------+



SQL result identical to the DataFrame result of 2.7


Both front-ends go through the same Catalyst optimizer. The raw `explain()` output is several hundred lines here (adaptive execution prints the initial and final plans, and the cached relation is expanded each time it is scanned), so the helper below keeps only the operator names of the final physical plan, for the DataFrame and the SQL version of the same question.

In [34]:
import contextlib, io, re

SKIP = {"Physical", "Final", "AdaptiveSparkPlan", "ResultQueryStage", "ShuffleQueryStage",
        "BroadcastQueryStage", "TableCacheQueryStage"}


def operators(d):
    """Operator names of a DataFrame's final physical plan, without the column lists."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        d.explain()
    final_plan = buf.getvalue().split("== Initial Plan ==")[0]
    matches = (re.match(r"^[\s+\-:*()\d]*([A-Z][A-Za-z]+)", line) for line in final_plan.splitlines())
    return [m.group(1) for m in matches if m and m.group(1) not in SKIP]


df_version = (jan.join(F.broadcast(pu), "PULocationID", "left")
                 .groupBy("PU_Borough").agg(F.count("*").alias("pickups")).orderBy(F.desc("pickups")))
sql_version = spark.sql("""
    SELECT z.Borough AS PU_Borough, COUNT(*) AS pickups
    FROM trips t LEFT JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough ORDER BY pickups DESC
""")
df_version.collect(); sql_version.collect()   # run both so that adaptive execution has a final plan

print("DataFrame :", " > ".join(operators(df_version)))
print("SQL       :", " > ".join(operators(sql_version)))

DataFrame : Sort > AQEShuffleRead > Exchange > HashAggregate > AQEShuffleRead > Exchange > HashAggregate > Project > BroadcastHashJoin > Project > Filter > InMemoryTableScan > InMemoryRelation > Project > Project > Project > ColumnarToRow > FileScan > BroadcastExchange > Project > Filter > FileScan
SQL       : Sort > AQEShuffleRead > Exchange > HashAggregate > AQEShuffleRead > Exchange > HashAggregate > Project > BroadcastHashJoin > Project > Filter > InMemoryTableScan > InMemoryRelation > Project > Project > Project > ColumnarToRow > FileScan > BroadcastExchange > Filter > FileScan


Same strategy on both sides: a `BroadcastHashJoin` (the lookup is broadcast whether or not the hint is given, since it is far below `spark.sql.autoBroadcastJoinThreshold`), a two-phase `HashAggregate` around an `Exchange`, and a final `Sort`. The only difference is a column-pruning `Project` on the broadcast side that the SQL planner folds directly into the CSV scan.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing

### 4.1 The whole of 2019: which season is busiest?

The eleven other monthly files (~1.1 GB in total) are downloaded to `/home/jovyan/data_extra` when that directory is mounted in the container, otherwise to `data/2019/` — the point is to keep this bulk out of the lab folder. Each file is filtered to its own month before the union, because every monthly file contains a few stray timestamps from other months.

The 84 M rows are scanned **once**: the per-month aggregate (counts and sums) is tiny, so it is cached, and the seasons are derived from it rather than from a second pass over the parquet files. (A first version recomputed the seasonal aggregate for every figure and ran the driver out of heap.)

Seasons are calendar seasons (winter = Dec, Jan, Feb of the same year). They do not have the same number of days, so the comparison is made on **trips per day**.

In [35]:
EXTRA_DIR = Path("/home/jovyan/data_extra") if Path("/home/jovyan/data_extra").is_dir() else DATA_DIR / "2019"
print("extra 2019 months stored in:", EXTRA_DIR.resolve())

keep = ["tpep_pickup_datetime", "trip_distance", "fare_amount"]
year_2019 = None
for m in range(1, 13):
    path = jan_2019_trip_data if m == 1 else download(f"trip-data/yellow_tripdata_2019-{m:02d}.parquet", EXTRA_DIR)
    part = (spark.read.parquet(str(path)).select(*keep)
                 .filter((F.year("tpep_pickup_datetime") == 2019) & (F.month("tpep_pickup_datetime") == m))
                 .withColumn("month", F.lit(m)))
    year_2019 = part if year_2019 is None else year_2019.unionByName(part)

# counts and sums per month: 12 rows, cached -> a single pass over the 84 M rows for the whole section
monthly = (year_2019.groupBy("month").agg(
        F.count("*").alias("trips"),
        F.sum("trip_distance").alias("sum_distance"),
        F.sum("fare_amount").alias("sum_fare"))
    .withColumn("avg_distance_mi", F.round(F.col("sum_distance") / F.col("trips"), 2))
    .withColumn("avg_fare", F.round(F.col("sum_fare") / F.col("trips"), 2))
    .orderBy("month")
    .cache())
monthly.select("month", "trips", "avg_distance_mi", "avg_fare").show(12)

extra 2019 months stored in: /home/jovyan/data_extra


+-----+-------+---------------+--------+
|month|  trips|avg_distance_mi|avg_fare|
+-----+-------+---------------+--------+
|    1|7696080|           2.83|   12.53|
|    2|7048744|           2.92|   12.88|
|    3|7866110|           3.02|    13.3|
|    4|7475636|           3.04|   13.37|
|    5|7598068|           3.06|   13.57|
|    6|6971026|           3.11|   13.77|
|    7|6310134|           3.11|   13.61|
|    8|6072851|           3.16|   13.69|
|    9|6567396|           3.09|   13.81|
|   10|7213588|           3.02|   13.63|
|   11|6877463|           2.93|   13.37|
|   12|6895933|           2.97|   13.59|
+-----+-------+---------------+--------+



In [36]:
season = (F.when(F.col("month").isin(12, 1, 2), "winter")
           .when(F.col("month").isin(3, 4, 5), "spring")
           .when(F.col("month").isin(6, 7, 8), "summer")
           .otherwise("fall"))
days_in_season = (F.when(F.col("season") == "winter", 31 + 28 + 31)
                   .when(F.col("season") == "spring", 31 + 30 + 31)
                   .when(F.col("season") == "summer", 30 + 31 + 31)
                   .otherwise(30 + 31 + 30))

seasonal = (monthly.withColumn("season", season)
                   .groupBy("season").agg(F.sum("trips").alias("trips"),
                                          F.sum("sum_distance").alias("sum_distance"),
                                          F.sum("sum_fare").alias("sum_fare"))
                   .withColumn("avg_distance_mi", F.round(F.col("sum_distance") / F.col("trips"), 2))
                   .withColumn("avg_fare", F.round(F.col("sum_fare") / F.col("trips"), 2))
                   .withColumn("days", days_in_season)
                   .withColumn("trips_per_day", F.round(F.col("trips") / F.col("days")))
                   .select("season", "trips", "avg_distance_mi", "avg_fare", "days", "trips_per_day")
                   .orderBy(F.desc("trips_per_day")))
seasonal.show()

+------+--------+---------------+--------+----+-------------+
|season|   trips|avg_distance_mi|avg_fare|days|trips_per_day|
+------+--------+---------------+--------+----+-------------+
|spring|22939814|           3.04|   13.41|  92|     249346.0|
|winter|21640757|            2.9|   12.98|  90|     240453.0|
|  fall|20658447|           3.01|    13.6|  91|     227016.0|
|summer|19354011|           3.13|   13.69|  92|     210370.0|
+------+--------+---------------+--------+----+-------------+



**Busiest season of 2019: spring, with 249,346 trips per day** (22.9 M trips over 92 days), then winter (240 k/day), fall (227 k/day) and **summer last (210 k/day)**. Month by month the peak is March (7.87 M) and the trough August (6.07 M, −23 %): New Yorkers leave town in August, and heat and holidays thin the business traffic that feeds the cabs. Summer trips are, however, the longest (3.13 miles on average) and the most expensive ($13.69): fewer commuters, proportionally more airport and leisure rides. Winter has the shortest trips (2.9 miles).

Using trips per day rather than totals matters: on raw totals spring still wins, but winter (90 days) would look further behind fall (91) and summer (92) than it really is.

### 4.2 Visualisations with Spark 4's native plotting

Since Spark 4.0 a DataFrame has a `.plot` accessor (backed by Plotly). For bar/line charts Spark collects the top `spark.sql.pyspark.plotting.max_rows` rows (1000 by default) — fine for aggregated data; for scatter plots it draws a random sample, so the scatter below is done directly on the 5 M-row credit-card DataFrame without a `toPandas()`.

In [37]:
# 1) Question 1.6 revisited: trips per hour of the day, January 2019 vs January 2026
per_hour_2026 = (df_2026.filter((F.col("tpep_pickup_datetime") >= "2026-01-01") & (F.col("tpep_pickup_datetime") < "2026-02-01"))
                        .groupBy(F.hour("tpep_pickup_datetime").alias("pickup_hour"))
                        .agg(F.count("*").alias("trips_2026")))
hourly = (per_hour.withColumnRenamed("trips", "trips_2019")
                  .join(per_hour_2026, "pickup_hour").orderBy("pickup_hour"))

hourly.plot.bar(x="pickup_hour", y=["trips_2019", "trips_2026"], barmode="group",
                title="Yellow-taxi pickups per hour of the day — January 2019 vs January 2026",
                labels={"value": "trips", "pickup_hour": "hour of the day"})

In [38]:
# 2) Question 1.5 revisited: trips per day in January 2019 (weekly rhythm, holidays, weather)
per_day.orderBy("pickup_date").plot.line(
    x="pickup_date", y="trips", markers=True,
    title="Yellow-taxi pickups per day — January 2019",
    labels={"pickup_date": "day", "trips": "trips"})

In [39]:
# 3) Questions 2.3 and 2.7 revisited: pickups and average fare by pickup borough
fig = (by_borough.join(fare_by_borough.select(F.col("PU_Borough").alias("borough"), "avg_fare"), "borough")
                 .orderBy(F.desc("pickups"))
                 .plot.bar(x="borough", y="pickups", color="avg_fare",
                           title="Pickups by borough (colour = average fare, $) — January 2019"))
fig

In [40]:
# 4) Question 4.1: trips per day by season of 2019
seasonal.orderBy("trips_per_day").plot.bar(
    x="season", y="trips_per_day", color="avg_fare",
    title="Yellow-taxi trips per day by season — 2019 (colour = average fare, $)")

In [41]:
# 5) Question 1.8 revisited: tip vs distance on credit-card trips.
# .plot.scatter samples the DataFrame itself (spark.sql.pyspark.plotting.sample_ratio), no toPandas() needed.
(card.filter((F.col("trip_distance") <= 30) & (F.col("tip_amount") <= 40))
     .select("trip_distance", "tip_amount", "passenger_count")
     .plot.scatter(x="trip_distance", y="tip_amount", color="passenger_count", opacity=0.5,
                   title="Tip vs trip distance, credit-card trips (random sample) — January 2019"))

Five figures, all drawn with `DataFrame.plot` (Plotly backend), without any `toPandas()` — Spark itself limits the rows it sends to the browser:

1. **Hourly profile, 2019 vs 2026** (question 1.6): the two Januaries side by side, hour by hour. Same daily rhythm (trough at 4 am, peak at 6 pm in both years) at roughly half the volume — and the loss is not uniform: the **morning rush (6–8 am) is down to 36–40 % of its 2019 level**, evenings and nights hold at 50–54 %. Hybrid work removed commuters; nightlife came back.
2. **Daily trips in January 2019** (1.5): the weekly saw-tooth (Friday peaks, Sunday troughs) with the 1 January and 21 January dips clearly visible.
3. **Pickups by borough, coloured by average fare** (2.3 and 2.7): the 90 % Manhattan bar next to the small but expensive Queens / Newark bars.
4. **Trips per day by season** (4.1).
5. **Tip vs distance on a random sample of credit-card trips** (1.8): the points line up along straight rays — the 20 / 25 / 30 % presets applied to a fare that is itself roughly linear in distance — with a row of zero-tip points along the bottom that gets denser as the distance grows.

### 4.3 A dataset of my choosing: green taxis, same month

The TLC also publishes the **green** (Boro) taxi records. Green cabs were created in 2013 precisely because yellow cabs rarely pick up outside Manhattan: they may only take street hails in the outer boroughs and northern Manhattan. The same zone join on the green January 2019 file tests whether the data shows this policy.

In [42]:
green_file = download("trip-data/green_tripdata_2019-01.parquet")
green = spark.read.parquet(str(green_file))
print(f"{green.count():,} green trips; pickup column is `lpep_pickup_datetime` (l = LPEP, the green-cab meter standard)")

green_jan = (green.filter((F.col("lpep_pickup_datetime") >= "2019-01-01") & (F.col("lpep_pickup_datetime") < "2019-02-01"))
                  .join(F.broadcast(pu), "PULocationID", "left"))
n_green = green_jan.count()

green_by_borough = (green_jan.groupBy(F.col("PU_Borough").alias("borough"))
                             .agg(F.count("*").alias("green_pickups"),
                                  F.round(100 * F.count("*") / n_green, 2).alias("green_share_pct")))

yellow_vs_green = (by_borough.select("borough", F.col("pickups").alias("yellow_pickups"), F.col("pickup_share_pct").alias("yellow_share_pct"))
                             .join(green_by_borough, "borough", "full").fillna(0)
                             .orderBy(F.desc("yellow_pickups")))
yellow_vs_green.show()

yellow_vs_green.plot.bar(x="borough", y=["yellow_share_pct", "green_share_pct"], barmode="group",
                         title="Share of pickups by borough — yellow vs green taxis, January 2019",
                         labels={"value": "% of the fleet's pickups"})

672,105 green trips; pickup column is `lpep_pickup_datetime` (l = LPEP, the green-cab meter standard)


+-------------+--------------+----------------+-------------+---------------+
|      borough|yellow_pickups|yellow_share_pct|green_pickups|green_share_pct|
+-------------+--------------+----------------+-------------+---------------+
|    Manhattan|       6950511|           90.31|       207728|          30.91|
|       Queens|        471113|            6.12|       191330|          28.47|
|      Unknown|        159807|            2.08|         1123|           0.17|
|     Brooklyn|         91896|            1.19|       215169|          32.02|
|        Bronx|         18056|            0.23|        56097|           8.35|
|          N/A|          3890|            0.05|          204|           0.03|
|          EWR|           446|            0.01|            8|            0.0|
|Staten Island|           361|             0.0|          333|           0.05|
+-------------+--------------+----------------+-------------+---------------+



**The policy is visible in the data.** Green cabs made 672 k trips in January 2019, eleven times fewer than yellow cabs, but their pickups are spread across the city: **Brooklyn 32 %, Manhattan 31 % (only the northern part, where green cabs may street-hail), Queens 28 %, Bronx 8 %**, against 90 % Manhattan for the yellow fleet. In absolute numbers green cabs pick up **2.3× more passengers in Brooklyn (215 k vs 92 k) and 3.1× more in the Bronx (56 k vs 18 k)** than yellow cabs do. The two datasets together describe one service split by geography, which the yellow file alone would never show.

## Clean-up

In [43]:
df.unpersist()
spark.stop()
print("Spark session stopped")

Spark session stopped
